In [2]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from scipy.stats import spearmanr

_here = Path.cwd()
if not (_here / 'salary_analysis.ipynb').exists():
    for _c in [_here / 'priors',
               _here / 'premier_league' / 'priors',
               _here / 'non_penalty_bayes' / 'premier_league' / 'priors',
               _here / 'team_strength' / 'non_penalty_bayes' / 'premier_league' / 'priors']:
        if _c.exists():
            os.chdir(_c)
            break

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT    = str(NOTEBOOK_DIR.parents[5])
OUT_DIR      = NOTEBOOK_DIR / 'outputs'
OUT_DIR.mkdir(exist_ok=True)

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

from salary_and_champ_regressions import (
    pl_team_season_goals, build_champ_panel, build_salary_panel,
)

# ── shared palette (one system, used consistently across every chart) ──────
C_XG        = '#2a78d6'   # xG series (categorical slot 1 / diverging "under" pole)
C_GOALS     = '#eb6834'   # goals series (categorical slot 2)
C_PRED      = '#1baf7a'   # predicted-as-its-own-series (categorical slot 3)
C_OVER      = '#e34948'   # over-prediction / positive bias / positive luck
C_UNDER     = '#2a78d6'   # under-prediction / negative bias / negative luck
C_GRID      = '#e1e0d9'
C_AXIS      = '#c3c2b7'
C_MUTED     = '#898781'
C_INK       = '#0b0b0b'
C_INK2      = '#52514e'

plt.rcParams.update({
    'figure.dpi': 150,
    'font.family': 'sans-serif',
    'axes.edgecolor': C_AXIS,
    'axes.labelcolor': C_INK,
    'text.color': C_INK,
    'xtick.color': C_INK2,
    'ytick.color': C_INK2,
    'axes.grid': True,
    'grid.color': C_GRID,
    'grid.linewidth': 0.8,
    'axes.axisbelow': True,
})

# ── pull in the salary / championship-form / PL-scoring panels ─────────────
raw = pd.read_csv(NOTEBOOK_DIR / 'raw_results.csv')
pl_goals = pl_team_season_goals(raw)

champ_perf = pd.read_csv(NOTEBOOK_DIR / 'champ_xg_history.csv')

champ_perf.head()

,season,team_id,team,xg,matches_played,goals,xga,goals_against,possession
0,2020/2021,9850,Norwich City,78.9,46,75.0,52.6,36.0,60.7
1,2020/2021,9937,Brentford,76.1,46,79.0,39.9,42.0,55.6
2,2020/2021,8655,Blackburn Rovers,69.9,46,65.0,54.1,54.0,57.6
3,2020/2021,9817,Watford,67.8,46,63.0,46.3,30.0,54.0
4,2020/2021,8678,AFC Bournemouth,65.3,46,73.0,50.6,46.0,58.0


- do regression for xG and poss for next season goals and conceded (champ and prem)
- find the difference between expected poitns and actual
- explore adding the possession adjustment from John Knight